# OTX Multi-Object Tracking Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omkar-334/otx/blob/gsoc-tracker/demo.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-omkar--334%2Fotx-blue?logo=github)](https://github.com/omkar-334/otx/tree/gsoc-tracker)

> **Colab:** Run the install cell, restart runtime, continue from Section 1. **Local:** Skip install.

## 0. Install (Colab only — skip locally)

In [ ]:
!pip install -q "git+https://github.com/omkar-334/otx.git@gsoc-tracker#subdirectory=lib"
!pip install -q lap cython_bbox

# Restart runtime after install
import os
os.kill(os.getpid(), 9)

## 1. Config & Setup

In [ ]:
# ============================================================
# CONFIG
# ============================================================
MODE = "pretrained"      # "pretrained" (COCO-80, instant) or "trained" (trains on COCO subset)
TRACK_THRESH = 0.5       # Min confidence for tracking association (also syncs model threshold)
VIS_THRESH = 0.3         # Min confidence for drawing boxes on video (cosmetic only)
VIDEO_NAME = "vehicles.mp4"  # Which video to track (see VIDEOS list below)
# ============================================================

import subprocess, cv2, torch
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

# COCO 80-class names for labeling boxes
COCO_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck",
    "boat", "traffic light", "fire hydrant", "stop sign", "parking meter", "bench",
    "bird", "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra",
    "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
    "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove",
    "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
    "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
    "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
    "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse",
    "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink",
    "refrigerator", "book", "clock", "vase", "scissors", "teddy bear",
    "hair drier", "toothbrush",
]

# Download demo videos from GitHub release
VIDEOS = ["vehicles.mp4", "bikes-1.mp4", "bikes-2.mp4", "jets-1.mp4", "jets-2.mp4", "apples.mp4", "suitcases.mp4"]
BASE_URL = "https://github.com/omkar-334/otx/releases/download/demo-videos"

video_dir = Path("videos")
video_dir.mkdir(exist_ok=True)

for name in VIDEOS:
    path = video_dir / name
    if not path.exists():
        print(f"Downloading {name}...")
        subprocess.run(["wget", "-q", "-O", str(path), f"{BASE_URL}/{name}"], check=True)

video_path = video_dir / VIDEO_NAME
cap = cv2.VideoCapture(str(video_path))
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps, total = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f"\nVideo: {VIDEO_NAME} ({w}x{h}, {fps:.0f} fps, {total} frames, {total/fps:.1f}s)")
print(f"Available: {VIDEOS}")
print(f"Mode: {MODE} | Device: {DEVICE}")

## 2. Load Detector & Tracker

**Detector parameters (`YOLOX`):**
- `label_info=80` — number of COCO classes
- `input_size=(416, 416)` — model input resolution (tiny=416, s/l=640)
- `mean/std` — ImageNet normalization values
- `model_name` — `yolox_tiny` (only variant with OTX-compatible pretrained weights)

**Tracker parameters (`ByteTrack`):**
- `track_thresh` — min confidence for first-stage association; also auto-syncs the model's internal filter
- `track_buffer=30` — frames to keep lost tracks alive before removal
- `match_thresh=0.8` — IoU threshold for matching detections to tracks (higher = stricter)

In [ ]:
from otx.backend.native.models.detection.yolox import YOLOX
from otx.backend.native.models.base import DataInputParams
from otx.backend.native.models.tracking.bytetrack import ByteTrack

if MODE == "pretrained":
    model = YOLOX(
        label_info=80,
        data_input_params=DataInputParams(
            input_size=(416, 416),
            mean=(123.675, 116.28, 103.53),
            std=(58.395, 57.12, 57.375),
        ),
        model_name="yolox_tiny",
    )
    model.eval()
    model.to(DEVICE)
    class_names = COCO_CLASSES
    print(f"Loaded pretrained YOLOX-Tiny ({model.num_classes} COCO classes)")

elif MODE == "trained":
    import json, random
    from pycocotools.coco import COCO

    CLASSES = [
        "person", "bicycle", "car", "motorcycle", "airplane",
        "bus", "truck", "cat", "dog", "sports ball",
        "horse", "elephant", "bear", "zebra", "giraffe",
        "backpack", "umbrella", "handbag", "suitcase", "bottle",
    ]

    COCO_ROOT = Path("data/coco")
    SUBSET_ROOT = Path("data/coco_subset")
    COCO_ROOT.mkdir(parents=True, exist_ok=True)

    for name, url in {
        "annotations": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
        "val2017": "http://images.cocodataset.org/zips/val2017.zip",
    }.items():
        extracted = COCO_ROOT / ("annotations" if name == "annotations" else name)
        if extracted.exists():
            print(f"{name} exists, skipping.")
            continue
        target = COCO_ROOT / f"{name}.zip"
        print(f"Downloading {name}...")
        subprocess.run(["wget", "-q", "-O", str(target), url], check=True)
        subprocess.run(["unzip", "-q", "-o", str(target), "-d", str(COCO_ROOT)], check=True)
        target.unlink()

    ann_file = COCO_ROOT / "annotations" / "instances_val2017.json"
    coco = COCO(str(ann_file))
    cat_ids = coco.getCatIds(catNms=CLASSES)
    cats = coco.loadCats(cat_ids)
    img_ids = sorted({id for cat_id in cat_ids for id in coco.getImgIds(catIds=[cat_id])})
    random.seed(42)
    random.shuffle(img_ids)

    old_to_new = {old_id: i + 1 for i, old_id in enumerate(sorted(cat_ids))}
    new_cats = [dict(c, id=old_to_new[c["id"]]) for c in sorted(cats, key=lambda c: c["id"])]
    (SUBSET_ROOT / "images").mkdir(parents=True, exist_ok=True)
    (SUBSET_ROOT / "annotations").mkdir(parents=True, exist_ok=True)

    for split, ids in {"train": img_ids[:800], "val": img_ids[800:1000]}.items():
        anns = coco.loadAnns(coco.getAnnIds(imgIds=ids, catIds=cat_ids, iscrowd=False))
        imgs = coco.loadImgs(ids)
        new_anns = [dict(a, category_id=old_to_new[a["category_id"]]) for a in anns]
        with open(SUBSET_ROOT / "annotations" / f"instances_{split}.json", "w") as f:
            json.dump({"images": imgs, "annotations": new_anns, "categories": new_cats}, f)
        out_img_dir = SUBSET_ROOT / "images" / split
        if not out_img_dir.exists():
            out_img_dir.symlink_to((COCO_ROOT / "val2017").resolve())
        print(f"[{split}] {len(ids)} images, {len(new_anns)} annotations")

    from otx.backend.native.engine import OTXEngine
    engine = OTXEngine.from_model_name(
        model_name="yolox_tiny", task="DETECTION",
        data_root=str(SUBSET_ROOT), work_dir="./train-workspace",
    )
    metrics = engine.train(max_epochs=10, seed=42, deterministic=False)
    print(f"Training complete! Metrics: {metrics}")

    model = engine.model
    model.eval()
    model.to(DEVICE)
    class_names = CLASSES
    print(f"Loaded trained YOLOX-Tiny ({model.num_classes} classes)")

# Tracker — detector-agnostic, no weights
tracker = ByteTrack(
    track_thresh=TRACK_THRESH,  # min confidence for association (auto-syncs model threshold)
    track_buffer=30,            # frames to keep lost tracks
    match_thresh=0.8,           # IoU threshold for matching
)

print(f"\nDetector: {model.model_name} ({model.num_classes} classes)")
print(f"Tracker:  {tracker.__class__.__name__} (track_thresh={tracker.track_thresh})")
print(f"Device:   {DEVICE}")

## 3. Track a Video

`tracker.track(model, video)` — full pipeline: read frames -> detect -> ByteTrack associate -> draw bboxes with track IDs -> save H.264 video.

In [ ]:
from IPython.display import Video, display

out = tracker.track(
    model,
    str(video_path),
    output_dir="videos/tracked",
    device=DEVICE,
    conf_thresh=VIS_THRESH,      # cosmetic: which boxes to draw
    class_names=class_names,     # label boxes with class names
    verbose=True,
)
display(Video(str(out), embed=True, width=640))

## 4. Single-Frame API

`track_frame()` returns an `OTXPredBatch` with `track_ids` — the same entity the OTX pipeline consumes, so tracking plugs directly into the existing data flow.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from otx.backend.native.tools.video import draw_detections
from otx.data.entity.torch import OTXPredBatch

tracker.reset()
cap = cv2.VideoCapture(str(video_path))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, ax in enumerate(axes):
    for _ in range(20):
        ret, frame = cap.read()
    if not ret:
        break

    preds = tracker.track_frame(model, frame, device=DEVICE)

    if i == 0:
        print(f"Return type: {type(preds).__name__}")
        print(f"  .bboxes:    {preds.bboxes[0].shape}")
        print(f"  .scores:    {preds.scores[0].shape}")
        print(f"  .labels:    {preds.labels[0].shape}")
        print(f"  .track_ids: {preds.track_ids[0]}")

    draw_detections(
        frame,
        preds.bboxes[0].cpu().numpy(),
        preds.scores[0].cpu().numpy(),
        labels=preds.labels[0].cpu().numpy(),
        track_ids=preds.track_ids[0].cpu().numpy(),
        class_names=class_names,
        conf_thresh=VIS_THRESH,
    )
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {(i + 1) * 20}")
    ax.axis("off")

cap.release()
plt.suptitle("track_frame() API — returns OTXPredBatch with track_ids", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Detection vs Tracking — Side by Side

Same frame: detection only (no persistent IDs) vs detection + ByteTrack (consistent track IDs across frames).

In [ ]:
import matplotlib.pyplot as plt
from otx.backend.native.tools.video import preprocess_frame, draw_detections

tracker.reset()
cap = cv2.VideoCapture(str(video_path))
for _ in range(60):
    ret, frame = cap.read()
cap.release()

# --- Detection only ---
frame_det = frame.copy()
batch = preprocess_frame(frame_det, model, DEVICE)
with torch.no_grad():
    det_preds = model.predict_step(batch, batch_idx=0)
draw_detections(frame_det, det_preds.bboxes[0].cpu().numpy(),
                det_preds.scores[0].cpu().numpy(),
                labels=det_preds.labels[0].cpu().numpy(),
                class_names=class_names, conf_thresh=VIS_THRESH)

# --- Detection + Tracking ---
frame_trk = frame.copy()
trk_preds = tracker.track_frame(model, frame_trk, device=DEVICE)
draw_detections(frame_trk, trk_preds.bboxes[0].cpu().numpy(),
                trk_preds.scores[0].cpu().numpy(),
                labels=trk_preds.labels[0].cpu().numpy(),
                track_ids=trk_preds.track_ids[0].cpu().numpy(),
                class_names=class_names, conf_thresh=VIS_THRESH)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
ax1.imshow(cv2.cvtColor(frame_det, cv2.COLOR_BGR2RGB))
ax1.set_title("Detection Only (no persistent IDs)")
ax1.axis("off")
ax2.imshow(cv2.cvtColor(frame_trk, cv2.COLOR_BGR2RGB))
ax2.set_title("Detection + ByteTrack (consistent track IDs)")
ax2.axis("off")
plt.tight_layout()
plt.show()

## 6. Export to OpenVINO IR & Track with Exported Model

Export the PyTorch detector to OpenVINO IR, then run tracking with the exported model — demonstrating the tracker works across inference backends.

In [ ]:
from pathlib import Path
from otx.types.export import OTXExportFormatType
from otx.types.precision import OTXPrecisionType

export_dir = Path("exported_models")
export_dir.mkdir(exist_ok=True)

exported_path = model.export(
    output_dir=export_dir,
    base_name="yolox_tiny_tracking",
    export_format=OTXExportFormatType.OPENVINO,
    precision=OTXPrecisionType.FP32,
)

print(f"Exported to: {exported_path}")
for f in sorted(export_dir.glob("yolox_tiny_tracking*")):
    print(f"  {f.name} ({f.stat().st_size // 1024} KB)")

In [ ]:
from otx.backend.openvino.models.detection import OVDetectionModel

# Load the exported OpenVINO IR model
# OVDetectionModel.predict_step() handles normalization and bbox rescaling natively —
# no manual shim needed
ov_model = OVDetectionModel(model_path=exported_path, async_inference=False)
ov_model.data_input_params = model.data_input_params

print(f"OpenVINO model loaded: {type(ov_model).__name__}")

In [ ]:
# Track with OpenVINO model — same tracker, same API
tracker.reset()
ov_out = tracker.track(
    ov_model,
    str(video_path),
    output_dir="videos/tracked-ov",
    device="cpu",
    conf_thresh=VIS_THRESH,
    class_names=class_names,
    verbose=True,
)
print(f"\nPyTorch output:  videos/tracked/")
print(f"OpenVINO output: {ov_out}")
display(Video(str(ov_out), embed=True, width=640))

## Architecture

```
User Code
  │
  ├── model = YOLOX(...)           # any OTX detector
  ├── tracker = ByteTrack(...)     # any OTX tracker (no weights)
  └── tracker.track(model, video)  # detection + tracking
         │
         ├── preprocess_frame()    ─── video.py (shared utilities)
         ├── model.predict_step()  ─── OTXDetectionModel / OVDetectionModel
         ├── tracker.update()      ─── OTXTracker (ByteTrack/OC-SORT/BoT-SORT)
         └── draw_detections()     ─── video.py
```

**Key design**: The tracker has no weights or training loop — it's a post-processing algorithm. Swap the detector by changing one line (`YOLOX` -> `RTDETR`, `SSD`, etc.). Works with both PyTorch and OpenVINO IR models.